# Public Data Preprocessing for RRWH

This notebook loads a CHIRPS rainfall archive, cleans the data, computes annual rainfall statistics, and saves the outputs using the public workflow modules. It is intentionally generic and does not depend on private study-area paths.

## Inputs

- `public_rrwh_workflow/config/paths.yaml`
- `public_rrwh_workflow/config/parameters.yaml`
- A CHIRPS daily NetCDF folder
- A reference raster such as roof/building footprint data for alignment

## Outputs

- `output/maps/annual_rainfall_chirps_native.nc`
- `output/maps/rainfall_skewness_chirps_native.nc`
- `output/maps/rainfall_heaviness_chirps_native.nc`
- Cached intermediate Zarr files in the external temp/output directories if configured

In [ ]:
from pathlib import Path
import os
import sys

import dask
from dask.diagnostics import ProgressBar
import xarray as xr

WORKFLOW_ROOT = Path(__file__).resolve().parents[1] if '__file__' in globals() else Path.cwd() / 'public_rrwh_workflow'
SRC_DIR = WORKFLOW_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from preprocessing import load_config, load_paths, load_chirps, load_geotiff, align_to_reference, compute_rainfall_stats, clean_data

config = load_config(WORKFLOW_ROOT / 'config' / 'parameters.yaml')
paths = load_paths(WORKFLOW_ROOT / 'config' / 'paths.yaml')

output_root = Path(paths['output']['root'])
maps_dir = Path(paths['output']['maps'])
tables_dir = Path(paths['output']['tables'])
for folder in [output_root, maps_dir, tables_dir]:
    folder.mkdir(parents=True, exist_ok=True)

os.environ.setdefault('HDF5_USE_FILE_LOCKING', 'FALSE')

chirps_path = paths['data']['chirps']
reference_path = paths['data']['building_footprint']

print(f'Loading CHIRPS from: {chirps_path}')
chirps_ds = load_chirps(chirps_path)
chirps_clean = clean_data(chirps_ds)
print(f'CHIRPS shape: {chirps_clean.shape}')

print(f'Loading reference raster from: {reference_path}')
reference_da = load_geotiff(reference_path)
if reference_da.rio.crs is None:
    reference_da.rio.write_crs(config['crs']['projected'], inplace=True)

annual_rain, skewness, heaviness = compute_rainfall_stats(chirps_clean)

with ProgressBar():
    annual_rain = dask.compute(annual_rain)[0]
with ProgressBar():
    skewness = dask.compute(skewness)[0]
with ProgressBar():
    heaviness = dask.compute(heaviness)[0]

annual_aligned = align_to_reference(annual_rain, reference_da)
skew_aligned = align_to_reference(skewness, reference_da)
heavy_aligned = align_to_reference(heaviness, reference_da)

annual_path = maps_dir / 'annual_rainfall_chirps_native.nc'
skew_path = maps_dir / 'rainfall_skewness_chirps_native.nc'
heavy_path = maps_dir / 'rainfall_heaviness_chirps_native.nc'

annual_aligned.to_netcdf(annual_path)
skew_aligned.to_netcdf(skew_path)
heavy_aligned.to_netcdf(heavy_path)

print(f'Saved: {annual_path}')
print(f'Saved: {skew_path}')
print(f'Saved: {heavy_path}')